In [2]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
SEED = 42

In [3]:
from pathlib import Path

CURRENT_DIR = Path.cwd()

# Possible dataset locations depending on
# where VS Code starts the notebook
candidates = [
    CURRENT_DIR / "Data" / "UCI HAR Dataset",
    CURRENT_DIR.parent / "Data" / "UCI HAR Dataset"
]

DATA_DIR = None

for path in candidates:
    if path.exists():
        DATA_DIR = path
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate the UCI HAR Dataset folder."
    )

# DATA_DIR is:
# project/Data/UCI HAR Dataset
#
# Therefore project root is two levels above DATA_DIR
PROJECT_ROOT = DATA_DIR.parent.parent

# Folder where processed data will be saved
OUTPUT_DIR = PROJECT_ROOT / "processed_Data"

# Create it if it does not exist
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())
print("Output directory:", OUTPUT_DIR)
print("Output directory exists:", OUTPUT_DIR.exists())

Current directory: f:\research\Deep-Learning-Assignment\Preprocessing
Project root: f:\research\Deep-Learning-Assignment
Dataset directory: f:\research\Deep-Learning-Assignment\Data\UCI HAR Dataset
Dataset exists: True
Output directory: f:\research\Deep-Learning-Assignment\processed_Data
Output directory exists: True


In [4]:
SIGNALS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


def load_signals(base_path, split):
    signal_data = []

    for signal in SIGNALS:
        file_path = (
            base_path
            / split
            / "Inertial Signals"
            / f"{signal}_{split}.txt"
        )

        signal_data.append(
            np.loadtxt(file_path)
        )

    return np.transpose(
        np.array(signal_data),
        (1, 2, 0)
    )

In [5]:
ACTIVITY_NAMES = np.array([
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
])

In [6]:
X_train_full = load_signals(
    DATA_DIR,
    "train"
)

X_test = load_signals(
    DATA_DIR,
    "test"
)

print(X_train_full.shape)
print(X_test.shape)

(7352, 128, 9)
(2947, 128, 9)


In [7]:
y_train_full = np.loadtxt(
    DATA_DIR / "train" / "y_train.txt",
    dtype=int
)

y_test = np.loadtxt(
    DATA_DIR / "test" / "y_test.txt",
    dtype=int
)

In [8]:
y_train_full = y_train_full - 1
y_test = y_test - 1

print(np.unique(y_train_full))
print(np.unique(y_test))

[0 1 2 3 4 5]
[0 1 2 3 4 5]


In [9]:
subjects_train_full = np.loadtxt(
    DATA_DIR / "train" / "subject_train.txt",
    dtype=int
)

subjects_test = np.loadtxt(
    DATA_DIR / "test" / "subject_test.txt",
    dtype=int
)

In [10]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        X_train_full,
        y_train_full,
        groups=subjects_train_full
    )
)

In [11]:
X_train = X_train_full[train_idx]
X_val = X_train_full[val_idx]

y_train = y_train_full[train_idx]
y_val = y_train_full[val_idx]

subjects_train = subjects_train_full[train_idx]
subjects_val = subjects_train_full[val_idx]

In [12]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (5551, 128, 9)
Validation: (1801, 128, 9)
Testing: (2947, 128, 9)


In [13]:
train_subjects = set(
    np.unique(subjects_train)
)

val_subjects = set(
    np.unique(subjects_val)
)

test_subjects = set(
    np.unique(subjects_test)
)

print(
    "Train ∩ Validation:",
    train_subjects & val_subjects
)

print(
    "Train ∩ Test:",
    train_subjects & test_subjects
)

print(
    "Validation ∩ Test:",
    val_subjects & test_subjects
)

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()


In [14]:
assert train_subjects.isdisjoint(
    val_subjects
), "Train and validation subjects overlap"

assert train_subjects.isdisjoint(
    test_subjects
), "Train and test subjects overlap"

assert val_subjects.isdisjoint(
    test_subjects
), "Validation and test subjects overlap"

print(
    "Subject split verified: "
    "no overlap between train, validation and test."
)

Subject split verified: no overlap between train, validation and test.


In [15]:
def print_split_distribution(
    name,
    labels
):
    counts = np.bincount(
        labels,
        minlength=6
    )

    percentages = (
        counts /
        len(labels) *
        100
    )

    print(f"\n{name}")

    for i, activity in enumerate(
        ACTIVITY_NAMES
    ):
        print(
            f"{activity:20s} "
            f"{counts[i]:5d} "
            f"({percentages[i]:5.2f}%)"
        )


print_split_distribution(
    "Training",
    y_train
)

print_split_distribution(
    "Validation",
    y_val
)

print_split_distribution(
    "Testing",
    y_test
)


Training
Walking                888 (16.00%)
Walking Upstairs       797 (14.36%)
Walking Downstairs     744 (13.40%)
Sitting                993 (17.89%)
Standing              1053 (18.97%)
Laying                1076 (19.38%)

Validation
Walking                338 (18.77%)
Walking Upstairs       276 (15.32%)
Walking Downstairs     242 (13.44%)
Sitting                293 (16.27%)
Standing               321 (17.82%)
Laying                 331 (18.38%)

Testing
Walking                496 (16.83%)
Walking Upstairs       471 (15.98%)
Walking Downstairs     420 (14.25%)
Sitting                491 (16.66%)
Standing               532 (18.05%)
Laying                 537 (18.22%)


In [16]:
mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

std = np.where(
    std == 0,
    1,
    std
)

In [17]:
X_train_normalized = (
    X_train - mean
) / std

X_val_normalized = (
    X_val - mean
) / std

X_test_normalized = (
    X_test - mean
) / std

In [18]:
for name, X in [
    ("Train", X_train_normalized),
    ("Validation", X_val_normalized),
    ("Test", X_test_normalized)
]:
    assert not np.isnan(X).any(), (
        f"{name} contains NaN values"
    )

    assert not np.isinf(X).any(), (
        f"{name} contains infinite values"
    )

print(
    "Normalization verified: "
    "no NaN or infinite values."
)

Normalization verified: no NaN or infinite values.


In [19]:
X_train_normalized = (
    X_train_normalized.astype(
        np.float32
    )
)

X_val_normalized = (
    X_val_normalized.astype(
        np.float32
    )
)

X_test_normalized = (
    X_test_normalized.astype(
        np.float32
    )
)

y_train = y_train.astype(
    np.int32
)

y_val = y_val.astype(
    np.int32
)

y_test = y_test.astype(
    np.int32
)

In [20]:
print(
    "X train dtype:",
    X_train_normalized.dtype
)

print(
    "X validation dtype:",
    X_val_normalized.dtype
)

print(
    "X test dtype:",
    X_test_normalized.dtype
)

print(
    "y train dtype:",
    y_train.dtype
)

X train dtype: float32
X validation dtype: float32
X test dtype: float32
y train dtype: int32


In [21]:
print(
    "Training mean:",
    X_train_normalized.mean(
        axis=(0, 1)
    )
)

print(
    "Training std:",
    X_train_normalized.std(
        axis=(0, 1)
    )
)

Training mean: [ 6.7512917e-10 -6.8251133e-10 -1.3422051e-10 -2.7427962e-09
  1.8898247e-09  2.0938400e-10  8.6599075e-09  3.6003311e-08
  1.1537595e-08]
Training std: [1.         0.9999995  1.0000001  0.99999815 1.         0.99999887
 0.9999995  0.99999934 1.0000004 ]


In [22]:
ACTIVITY_NAMES = np.array([
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
])

output_file = (
    OUTPUT_DIR
    / "har_processed.npz"
)
SIGNAL_NAMES = np.array(
    SIGNALS
)

np.savez_compressed(
    output_file,

    X_train=X_train_normalized,
    X_val=X_val_normalized,
    X_test=X_test_normalized,

    y_train=y_train,
    y_val=y_val,
    y_test=y_test,

    subjects_train=subjects_train,
    subjects_val=subjects_val,
    subjects_test=subjects_test,

    mean=mean,
    std=std,

    activity_names=ACTIVITY_NAMES,
    signal_names=SIGNAL_NAMES
)

print(
    "Saved processed dataset to:",
    output_file
)


Saved processed dataset to: f:\research\Deep-Learning-Assignment\processed_Data\har_processed.npz


In [23]:
print("File exists:", output_file.exists())
print(
    "File size:",
    round(output_file.stat().st_size / (1024 * 1024), 2),
    "MB"
)

File exists: True
File size: 23.12 MB


In [24]:
check_data = np.load(output_file)

print(check_data.files)

print("X_train:", check_data["X_train"].shape)
print("X_val:", check_data["X_val"].shape)
print("X_test:", check_data["X_test"].shape)

print(
    "Train labels:",
    np.unique(check_data["y_train"])
)

print(
    "Validation labels:",
    np.unique(check_data["y_val"])
)

print(
    "Test labels:",
    np.unique(check_data["y_test"])
)

['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'subjects_train', 'subjects_val', 'subjects_test', 'mean', 'std', 'activity_names', 'signal_names']
X_train: (5551, 128, 9)
X_val: (1801, 128, 9)
X_test: (2947, 128, 9)
Train labels: [0 1 2 3 4 5]
Validation labels: [0 1 2 3 4 5]
Test labels: [0 1 2 3 4 5]
